In [2]:
import numpy as np
import pandas as pd
import os

# 1. Use a raw string (r"") to handle the backslashes in Windows paths
file_path = "tournament_model_ml.csv"

def load_data(path):
    # Check if the file actually exists at this location to debug
    if not os.path.exists(path):
        print(f"Error: The file was not found at {path}")
        return None
    return pd.read_csv(path)

# 2. Load the data
df = load_data(file_path)

if df is not None:
    # 3. Split into train (2015-2022) and test (2023-2025)
    train_data = df[(df['season'] >= 2015) & (df['season'] <= 2022)]
    test_data = df[(df['season'] >= 2023) & (df['season'] <= 2025)]

    # 4. Verify the split
    print(f"Training set: {train_data.shape[0]} rows (Seasons: {train_data['season'].unique()})")
    print(f"Testing set: {test_data.shape[0]} rows (Seasons: {test_data['season'].unique()})")

Training set: 754 rows (Seasons: [2015 2016 2017 2018 2019 2021 2022])
Testing set: 314 rows (Seasons: [2023 2024 2025])


In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score

X_train = train_data.drop(columns=['win_label', 'season'])
y_train = train_data['win_label']

X_test = test_data.drop(columns=['win_label', 'season'])
y_test = test_data['win_label']
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Initialize and Train the Logistic Regression Model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("--- Cross-Validation Results ---")
print(f"Scores for each fold: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")
print("--------------------------------\n")
# 6. Make Predictions on the test set
y_pred = model.predict(X_test_scaled)

# 7. Evaluate the model performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

--- Cross-Validation Results ---
Scores for each fold: [0.77483444 0.80794702 0.7615894  0.78807947 0.70666667]
Mean CV Accuracy: 0.7678
Standard Deviation: 0.0342
--------------------------------

Model Accuracy: 0.7580

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.76      0.76       157
           1       0.76      0.76      0.76       157

    accuracy                           0.76       314
   macro avg       0.76      0.76      0.76       314
weighted avg       0.76      0.76      0.76       314


Confusion Matrix:
[[119  38]
 [ 38 119]]


In [3]:
feature_names = X_train.columns

# 2. Get the coefficients from your trained logistic regression model
# model.coef_ returns a 2D array, so we access the first element [0]
coefficients = model.coef_[0]

# 3. Create a DataFrame to store feature names and their impact
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients) # Absolute value for ranking magnitude
})

# 4. Sort the variables by their importance (absolute magnitude)
importance_df = importance_df.sort_values(by='Abs_Coefficient', ascending=False)

print("Variable Importance (Ranked by Magnitude):")
print(importance_df[['Feature', 'Coefficient', 'Abs_Coefficient']])

Variable Importance (Ranked by Magnitude):
        Feature  Coefficient  Abs_Coefficient
4      SRS_diff     2.564142         2.564142
7     seed_diff     1.164645         1.164645
8  win_pct_diff     0.567837         0.567837
2      FG%_diff    -0.349191         0.349191
5      TOV_diff     0.297469         0.297469
6      TRB_diff     0.263793         0.263793
1      AST_diff     0.220047         0.220047
3      FT%_diff     0.187509         0.187509
0      3P%_diff     0.065506         0.065506
